In [1]:
import tensorflow as tf

In [2]:
from tensorflow.keras.layers import Input, Dense, SimpleRNN, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Things you should automatically know and have memorized
# N = number of samples
# T = sequence length
# D = number of input features
# M = number of hidden units
# K = number of output units

In [4]:
# make some dummy data
N = 1
T = 10
D = 3
K = 2
X = np.random.randn(N, T, D)

In [5]:
# make an RNN
M = 5 # number of hidden units
i = Input(shape=(T, D))
x = SimpleRNN(M)(i)
x = Dense(K)(x)

model = Model(i, x)

In [6]:
# get the output
Yhat = model.predict(X)
print(Yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step
[[0.06822848 0.9628967 ]]


In [7]:
# see if we can replicate this output
# get the weights first
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57 (228.00 B)

 Trainable params: 57 (228.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
# see what's returned
model.layers[1].get_weights()

[array([[-0.85018057,  0.7197892 , -0.40474316, -0.78097004,  0.56147414],
        [ 0.63606745,  0.28589016, -0.5116658 , -0.68963003,  0.37949258],
        [-0.6480105 , -0.03624034, -0.68445796,  0.5105851 , -0.35101795]],
       dtype=float32),
 array([[ 0.2303592 ,  0.9201868 ,  0.03578644, -0.16204897, -0.26953718],
        [-0.19609126,  0.11495033,  0.90656024,  0.3215894 ,  0.15186672],
        [-0.94540167,  0.22735828, -0.18972158, -0.13405961,  0.02361429],
        [ 0.11241609,  0.27373195, -0.21247102,  0.13448021,  0.92152303],
        [-0.04540011,  0.11585157, -0.30940214,  0.9133804 , -0.23350367]],
       dtype=float32),
 array([0., 0., 0., 0., 0.], dtype=float32)]

In [9]:
# check their shapes
# should make sense
# first output is input -> hidden
# second output is hidden -> hidden
# third output is bias term (vector of length M)
a,b,c = model.layers[1].get_weights()
print(a.shape, b.shape, c.shape)

(3, 5) (5, 5) (5,)


In [10]:
Wx, Wh, bh = model.layers[1].get_weights()
Wo, bo = model.layers[2].get_weights()

In [11]:
# manual RNN implementation
h_last = np.zeros(M) # initial hidden state
x = X[0] # the one and only sample
Yhats = [] # where we store the outputs

for t in range(T):
    h = np.tanh(x[t].dot(Wx) + h_last.dot(Wh) + bh)
    y = h.dot(Wo) + bo # we only care about this value on the last iteration
    Yhats.append(y)

    # important: assign h to h_last
    h_last = h
# print the final output
print(Yhats[-1])

[0.06822846 0.96289669]
